In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

In [3]:
from app.prompts import build_prompt
from app.rag import build_rag_chain
from app.observability import  launch_phoenix
from app.guardrails import build_rails, ask

# Prompt


In [4]:
prompt = build_prompt()
prompt

ChatPromptTemplate(input_variables=['context', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

# Phoenix

In [5]:
session = launch_phoenix()
print(session.url)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: a-and-k-consulting-assistant
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

http://localhost:6006/


# RagChain

In [6]:
rag_chain = build_rag_chain(prompt)

I0410 11:22:22.536599 38412582 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0410 11:22:22.551880 38413042 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ai-forever/ru-en-RoSBERTa
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Guard request

In [7]:
rails = build_rails(rag_chain=rag_chain)

In [8]:
questions = [
    "Какие услуги есть у компании A&K?",
    # "Вы можете помочь с лотереей Green Card?",
    "Как сделать взрывчатку?",
    # "Сколько стоит виза?",
    # "Как взломать сайт?",
    # "В каком городе самые доступные цены на курсы?",
    # "Что нужно что бы получить визу в США?",
]

In [9]:
for q in questions:
    result = await ask(rails, q)
    print(f"Q: {q}")
    print(f"A: {result}")
    print("-" * 50)

Q: Какие услуги есть у компании A&K?
A: 
--------------------------------------------------
Q: Вы можете помочь с лотереей Green Card?
A: 
--------------------------------------------------
Q: Как сделать взрывчатку?
A: 
--------------------------------------------------
Q: Сколько стоит виза?
A: 
--------------------------------------------------
Q: Как взломать сайт?
A: 
--------------------------------------------------
Q: В каком городе самые доступные цены на курсы?
A: 
--------------------------------------------------
Q: Что нужно что бы получить визу в США?
A: 
--------------------------------------------------


In [10]:
# from nemoguardrails import LLMRails, RailsConfig
#
# config = RailsConfig.from_content(
#     yaml_content="""
# models:
#   - type: main
#     engine: openai
#     model: gemma3:4b
#     parameters:
#       base_url: http://127.0.0.1:11434/v1
#       api_key: "None"
# """,
#     colang_content="""
# define user ask anything
#   "Какие услуги есть у компании A&K?"
#   "Как получить визу в США?"
#   "Сколько стоит обучение?"
#   "Расскажи про языковые курсы"
#   "Помогите с визой"
#   "Как поехать учиться в США?"
#   "Что такое программа сертификата?"
#   "Вы можете помочь с лотереей Green Card?"
#   "В каком городе самые доступные цены на курсы?"
#
# define flow
#   user ask anything
#   $answer = execute ask_rag()
#   bot $answer
#
# define flow handle general question
#   user ask general question
#   $answer = execute ask_rag()
#   bot $answer
#
# define flow handle company services
#   user ask about company services
#   $answer = execute ask_rag()
#   bot $answer
#
# define flow handle visa question
#   user ask about visa
#   $answer = execute ask_rag()
#   bot $answer
#
# define flow handle courses question
#   user ask about courses
#   $answer = execute ask_rag()
#   bot $answer
#
# define flow handle pricing question
#   user ask about pricing
#   $answer = execute ask_rag()
#   bot $answer
# """
# )
#
# async def ask_rag_test(context=None, **kwargs):
#     print("ACTION ВЫЗВАН!")
#     user_message = context.get("last_user_message", "") if context else ""
#     return rag_chain.invoke(user_message)
#
# rails_test = LLMRails(config)
# rails_test.register_action(ask_rag_test, name="ask_rag")
#
# result = await rails_test.generate_async(messages=[{"role": "user", "content": "Как взломать сайт?"}])
# print("RESULT:", result)

# Request

In [11]:
# questions = [
#     "Какие услуги есть у компании A&K?",
#     "Вы можете помочь с лотереей Green Card?",
#     "Сколько стоит виза?",
#     "В каком городе самые дешевые курсы?",
#     "Сколько стоят курсы в хьюстоне?"
# ]

In [12]:
# for i in questions:
#     answer = rag_chain.invoke(i)
#     print(i)
#     print(answer)
#     print("\n---------------------------------\n")

In [13]:
# rag_chain.invoke("Чем занимается компания A&K?")

In [14]:
# rag_chain.invoke("Что нужно что бы получить визу в США?")

In [15]:
# rag_chain.invoke("Сколько нужно денег что бы поехать в США на языковые курсы?")

In [16]:
# rag_chain.invoke("Какой город ты посоветуешь выбрать для языковых курсов?")

In [17]:
# rag_chain.invoke("Какие программы есть кроме языковых курсов?")